# COP-RFS: Underdetermined DOA Estimation & Multi-Target Tracking

**IEEE TSP 2026** — Choi, J.H.

This notebook demonstrates:
1. **6-method DOA comparison**: CBF, MVDR, MUSIC vs COP-CBF, COP-MVDR, COP-Subspace
2. **Underdetermined resolution**: K > M-1 sources with M sensors
3. **GM-PHD multi-target tracking** fed by COP-MVDR (no K needed)

Key insight: 4th-order cumulant creates virtual array (M=4 → M_v=7), enabling:
- Resolution of up to 6 sources (vs MUSIC's 3)
- Automatic Gaussian noise elimination
- K-free beamforming (COP-CBF, COP-MVDR)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DrJinHoChoi/IronDome-DOA-Tracking/blob/master/COP_RFS_DOA_Tracking.ipynb)

## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import toeplitz
import time

plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11

# Array parameters
M = 4           # Physical sensors
RHO = 2         # Cumulant order (4th-order)
M_V = RHO * (M - 1) + 1  # = 7, Virtual array size

print(f'Physical array: M = {M} sensors')
print(f'Virtual  array: M_v = {M_V} sensors (ρ={RHO})')
print(f'MUSIC max sources: {M-1}')
print(f'COP   max sources: {M_V-1}')

## 2. Core Functions

In [ ]:
N_ANGLES = 361
scan_deg = np.linspace(-90, 90, N_ANGLES)
scan_rad = np.radians(scan_deg)


def gen_signal(M, T, doas_deg, snr=20, signal_type='bpsk'):
    """Generate array signal with multiple DOA sources."""
    K = len(doas_deg)
    A = np.zeros((M, K), dtype=complex)
    for k in range(K):
        for m in range(M):
            A[m, k] = np.exp(1j * np.pi * m * np.sin(np.radians(doas_deg[k])))

    if signal_type == 'bpsk':
        S = (2 * np.random.randint(0, 2, (K, T)) - 1).astype(complex)
    elif signal_type == 'qpsk':
        S = np.exp(1j * np.pi / 4 * (2 * np.random.randint(0, 4, (K, T)) + 1))
    else:
        S = np.random.randn(K, T) + 1j * np.random.randn(K, T)

    S *= np.sqrt(10 ** (snr / 10))
    N = (np.random.randn(M, T) + 1j * np.random.randn(M, T)) / np.sqrt(2)
    return A @ S + N


def compute_cumulant(X, M=4):
    """4th-order cumulant Toeplitz matrix (sum co-array)."""
    T = X.shape[1]
    L = 2 * (M - 1)
    M_v = L + 1
    R = X @ X.conj().T / T
    c4 = np.zeros(M_v, dtype=complex)
    cnt = np.zeros(M_v)
    for i1 in range(M):
        for i2 in range(M):
            xi12 = X[i1] * X[i2]
            for i3 in range(M):
                xi3c = X[i3].conj()
                for i4 in range(M):
                    tau = (i1 + i2) - (i3 + i4)
                    if 0 <= tau <= L:
                        m4 = np.mean(xi12 * xi3c * X[i4].conj())
                        c4[tau] += m4 - R[i1, i3] * R[i2, i4] - R[i1, i4] * R[i2, i3]
                        cnt[tau] += 1
    mask = cnt > 0
    c4[mask] /= cnt[mask]
    return toeplitz(c4, c4.conj())

## 3. DOA Estimation Methods (6 algorithms)

In [ ]:
# === Physical Array Methods (M=4) ===

def cbf_spectrum(X):
    """Conventional Beamforming: P = a^H R a  (no K needed)"""
    R = X @ X.conj().T / X.shape[1]
    P = np.zeros(N_ANGLES)
    for i in range(N_ANGLES):
        a = np.exp(1j * np.pi * np.arange(M) * np.sin(scan_rad[i]))
        P[i] = np.real(a.conj() @ R @ a)
    return P / np.max(P)


def mvdr_spectrum(X):
    """MVDR (Capon): P = 1 / (a^H R^-1 a)  (no K needed)"""
    R = X @ X.conj().T / X.shape[1] + 1e-6 * np.eye(M)
    R_inv = np.linalg.inv(R)
    P = np.zeros(N_ANGLES)
    for i in range(N_ANGLES):
        a = np.exp(1j * np.pi * np.arange(M) * np.sin(scan_rad[i]))
        P[i] = 1.0 / max(np.real(a.conj() @ R_inv @ a), 1e-15)
    return P / np.max(P)


def music_spectrum(X, K):
    """MUSIC: P = 1 / (a^H U_n U_n^H a)  (K required)"""
    R = X @ X.conj().T / X.shape[1]
    eigvals, eigvecs = np.linalg.eigh(R)
    idx = np.argsort(np.abs(eigvals))[::-1]
    U_n = eigvecs[:, idx][:, min(K, M-1):]
    P = np.zeros(N_ANGLES)
    for i in range(N_ANGLES):
        a = np.exp(1j * np.pi * np.arange(M) * np.sin(scan_rad[i]))
        P[i] = 1.0 / max(np.real(np.sum(np.abs(U_n.conj().T @ a)**2)), 1e-15)
    return P / np.max(P)


# === COP Virtual Array Methods (M_v=7, noise-free) ===

def cop_cbf_spectrum(C):
    """COP-CBF: Conventional BF on virtual array  (NO K needed)
    P = a_v^H C a_v
    """
    P = np.zeros(N_ANGLES)
    for i in range(N_ANGLES):
        a_v = np.exp(1j * np.pi * np.arange(M_V) * np.sin(scan_rad[i]))
        P[i] = np.real(a_v.conj() @ C @ a_v)
    P = np.abs(P)
    return P / np.max(P)


def cop_mvdr_spectrum(C):
    """COP-MVDR: Capon BF on cumulant matrix  (NO K needed)
    P = 1 / (a_v^H C^-1 a_v)
    """
    C_inv = np.linalg.inv(C + 1e-6 * np.eye(M_V))
    P = np.zeros(N_ANGLES)
    for i in range(N_ANGLES):
        a_v = np.exp(1j * np.pi * np.arange(M_V) * np.sin(scan_rad[i]))
        P[i] = 1.0 / max(abs(np.real(a_v.conj() @ C_inv @ a_v)), 1e-15)
    P = np.abs(P)
    return P / np.max(P)


def cop_subspace_spectrum(C, K):
    """COP-Subspace: signal/noise subspace on cumulant  (K required)
    P = (a_v^H P_s a_v) / (a_v^H P_n a_v)
    """
    eigvals, eigvecs = np.linalg.eigh(C)
    idx = np.argsort(np.abs(eigvals))[::-1]
    eigvecs = eigvecs[:, idx]
    K_eff = min(K, M_V - 1)
    U_s, U_n = eigvecs[:, :K_eff], eigvecs[:, K_eff:]
    P = np.zeros(N_ANGLES)
    for i in range(N_ANGLES):
        a_v = np.exp(1j * np.pi * np.arange(M_V) * np.sin(scan_rad[i]))
        num = np.real(np.sum(np.abs(U_s.conj().T @ a_v)**2))
        den = np.real(np.sum(np.abs(U_n.conj().T @ a_v)**2))
        P[i] = num / max(den, 1e-15)
    return P / np.max(P)


def find_peaks(P, K, min_height=0.1, min_sep=5):
    """Find K highest peaks in spectrum."""
    peaks = []
    for i in range(2, N_ANGLES - 2):
        if P[i] > P[i-1] and P[i] > P[i+1] and P[i] > min_height:
            peaks.append((P[i], scan_deg[i]))
    peaks.sort(reverse=True)
    doas = sorted([p[1] for p in peaks[:K*2]])
    filt = []
    for d in doas:
        if not filt or abs(d - filt[-1]) > min_sep:
            filt.append(d)
    return filt[:K]

## 4. Experiment 1: Determined Case (K ≤ M-1)

In [ ]:
np.random.seed(42)
T = 256

cases = [
    ([-30, 30], '2 sources, 60° apart'),
    ([-15, 15], '2 sources, 30° apart'),
    ([-5, 5],   '2 sources, 10° apart (close!)'),
]

fig, axes = plt.subplots(len(cases), 1, figsize=(14, 4*len(cases)))

for idx, (doas, title) in enumerate(cases):
    ax = axes[idx]
    K = len(doas)
    X = gen_signal(M, T, doas, snr=20)
    C = compute_cumulant(X)

    methods = [
        ('CBF',          cbf_spectrum(X),            'gray',    '--', 1.0),
        ('MVDR',         mvdr_spectrum(X),           'orange',  '--', 1.5),
        ('MUSIC (K)',    music_spectrum(X, K),        'blue',    '-',  1.5),
        ('COP-CBF',      cop_cbf_spectrum(C),        'green',   '--', 1.5),
        ('COP-MVDR',     cop_mvdr_spectrum(C),       'red',     '-',  2.0),
        ('COP-Sub (K)',  cop_subspace_spectrum(C, K), 'darkred', '-',  2.0),
    ]

    for name, P, color, ls, lw in methods:
        ax.plot(scan_deg, 10*np.log10(P + 1e-10), color=color, ls=ls, lw=lw, label=name)

    for d in doas:
        ax.axvline(d, color='black', ls=':', alpha=0.5, lw=0.8)

    ax.set_title(f'{title}  (K={K} ≤ M-1={M-1})', fontweight='bold')
    ax.set_ylabel('Spectrum (dB)')
    ax.set_xlim(-90, 90)
    ax.set_ylim(-30, 3)
    ax.legend(loc='upper right', fontsize=9, ncol=3)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('DOA (degrees)')
plt.tight_layout()
plt.savefig('fig_determined.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_determined.png')

## 5. Experiment 2: Underdetermined Case (K > M-1) — COP Advantage

In [ ]:
np.random.seed(42)

cases_ud = [
    ([-40, 0, 40],          '3 sources (K=3 = M-1, boundary)'),
    ([-30, -10, 10, 30],    '4 sources (K=4 > M-1=3, UNDERDETERMINED)'),
    ([-50, -20, 10, 35, 60],'5 sources (K=5 >> M-1, UNDERDETERMINED)'),
]

fig, axes = plt.subplots(len(cases_ud), 1, figsize=(14, 4*len(cases_ud)))

for idx, (doas, title) in enumerate(cases_ud):
    ax = axes[idx]
    K = len(doas)
    X = gen_signal(M, T, doas, snr=20)
    C = compute_cumulant(X)

    methods = [
        ('CBF',          cbf_spectrum(X),            'gray',    '--', 1.0),
        ('MVDR',         mvdr_spectrum(X),           'orange',  '--', 1.5),
        ('MUSIC (K)',    music_spectrum(X, K),        'blue',    '-',  1.5),
        ('COP-CBF',      cop_cbf_spectrum(C),        'green',   '--', 1.5),
        ('COP-MVDR',     cop_mvdr_spectrum(C),       'red',     '-',  2.0),
        ('COP-Sub (K)',  cop_subspace_spectrum(C, K), 'darkred', '-',  2.0),
    ]

    for name, P, color, ls, lw in methods:
        ax.plot(scan_deg, 10*np.log10(P + 1e-10), color=color, ls=ls, lw=lw, label=name)

    for d in doas:
        ax.axvline(d, color='black', ls=':', alpha=0.5, lw=0.8)

    tag = ' << UNDERDETERMINED' if K > M-1 else ''
    ax.set_title(f'{title}{tag}', fontweight='bold',
                 color='red' if K > M-1 else 'black')
    ax.set_ylabel('Spectrum (dB)')
    ax.set_xlim(-90, 90)
    ax.set_ylim(-30, 3)
    ax.legend(loc='upper right', fontsize=9, ncol=3)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('DOA (degrees)')
plt.tight_layout()
plt.savefig('fig_underdetermined.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_underdetermined.png')

## 6. Experiment 3: RMSE vs SNR

In [ ]:
np.random.seed(42)
snrs = np.arange(-5, 31, 5)
T = 256
doas = [-20, 20]
K = len(doas)
N_MC = 50

method_names = ['CBF', 'MVDR', 'MUSIC', 'COP-CBF', 'COP-MVDR', 'COP-Sub']
rmse_all = {m: [] for m in method_names}

for snr in snrs:
    errs = {m: [] for m in method_names}
    for trial in range(N_MC):
        X = gen_signal(M, T, doas, snr=snr)
        C = compute_cumulant(X)

        spectra = {
            'CBF': cbf_spectrum(X),
            'MVDR': mvdr_spectrum(X),
            'MUSIC': music_spectrum(X, K),
            'COP-CBF': cop_cbf_spectrum(C),
            'COP-MVDR': cop_mvdr_spectrum(C),
            'COP-Sub': cop_subspace_spectrum(C, K),
        }

        for m, P in spectra.items():
            est = find_peaks(P, K)
            if len(est) >= K:
                err = [min(abs(e - t) for t in doas) for e in est[:K]]
                errs[m].append(np.mean(np.array(err)**2))
            else:
                errs[m].append(100)

    for m in method_names:
        rmse_all[m].append(np.sqrt(np.mean(errs[m])))

fig, ax = plt.subplots(figsize=(10, 6))
styles = {
    'CBF':      ('gray', '--', 'o'),
    'MVDR':     ('orange', '--', 's'),
    'MUSIC':    ('blue', '-', '^'),
    'COP-CBF':  ('green', '--', 'D'),
    'COP-MVDR': ('red', '-', 'v'),
    'COP-Sub':  ('darkred', '-', 'p'),
}
for m in method_names:
    c, ls, mk = styles[m]
    ax.semilogy(snrs, rmse_all[m], color=c, ls=ls, marker=mk, lw=2, label=m)

ax.set_xlabel('SNR (dB)')
ax.set_ylabel('RMSE (degrees)')
ax.set_title('DOA RMSE vs SNR — 6-Method Comparison (K=2, M=4)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fig_rmse_vs_snr.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_rmse_vs_snr.png')

## 7. Experiment 4: GM-PHD Multi-Target Tracking

In [ ]:
class SimplePHD:
    """Simplified GM-PHD tracker for DOA tracking."""
    def __init__(self, dt=0.1):
        self.dt = dt
        self.tracks = []  # [theta, theta_dot, weight, label]
        self._next = 1

    def update(self, doas_rad):
        new_tracks = []
        # Predict
        predicted = [[th + thd*self.dt, thd, w*0.9, lbl]
                     for th, thd, w, lbl in self.tracks]
        # Associate
        used = set()
        for pth, pthd, pw, plbl in predicted:
            best_j, best_d = -1, 999
            for j, doa in enumerate(doas_rad):
                if j not in used:
                    d = abs(pth - doa)
                    if d < np.radians(10) and d < best_d:
                        best_j, best_d = j, d
            if best_j >= 0:
                innov = doas_rad[best_j] - pth
                new_tracks.append([pth + 0.7*innov, pthd + 0.3*innov/self.dt,
                                   min(pw+0.2, 1.5), plbl])
                used.add(best_j)
            else:
                pw *= 0.7
                if pw > 0.05:
                    new_tracks.append([pth, pthd, pw, plbl])
        # Birth
        for j, doa in enumerate(doas_rad):
            if j not in used:
                new_tracks.append([doa, 0.0, 0.15, self._next])
                self._next += 1
        self.tracks = new_tracks

    def get_confirmed(self):
        return [(np.degrees(th), np.degrees(thd), w, lbl)
                for th, thd, w, lbl in self.tracks if w > 0.4]


# Trajectory: 2 targets converge, cross, diverge; 1 new target births
np.random.seed(42)
T = 256

trajectory = [
    [-40, 30],
    [-30, 25],
    [-20, 15],
    [-10, 10],
    [-5, 5],
    [0, 0],
    [5, -5],
    [15, -15, 45],     # new target births
    [25, -25, 45],
    [35, -35, 45],
]

tracker = SimplePHD(dt=1.0)
all_true, all_est, all_trk = [], [], []

for step, doas in enumerate(trajectory):
    K = len(doas)
    X = gen_signal(M, T, doas, snr=20)
    C = compute_cumulant(X)
    P = cop_mvdr_spectrum(C)
    est = find_peaks(P, K+1)
    tracker.update(np.radians(est))
    confirmed = tracker.get_confirmed()

    all_true.append(doas)
    all_est.append(est)
    all_trk.append([(d, lbl) for d, _, _, lbl in confirmed])

# Plot tracking result
fig, ax = plt.subplots(figsize=(12, 6))

for step, doas in enumerate(all_true):
    for d in doas:
        ax.plot(step, d, 'ko', ms=8, alpha=0.3)

for step, est in enumerate(all_est):
    for d in est:
        ax.plot(step, d, 'r+', ms=10, mew=2)

# Connect tracks
track_hist = {}
for step, trks in enumerate(all_trk):
    for d, lbl in trks:
        if lbl not in track_hist:
            track_hist[lbl] = ([], [])
        track_hist[lbl][0].append(step)
        track_hist[lbl][1].append(d)

colors = plt.cm.Set1(np.linspace(0, 1, len(track_hist)))
for i, (lbl, (steps, ds)) in enumerate(track_hist.items()):
    ax.plot(steps, ds, '-o', color=colors[i], lw=2, ms=6, label=f'Track {lbl}')

ax.set_xlabel('Time Step')
ax.set_ylabel('DOA (degrees)')
ax.set_title('GM-PHD Multi-Target Tracking (COP-MVDR, K-free)', fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
ax.axvline(7, color='green', ls='--', alpha=0.5, label='New target birth')
plt.tight_layout()
plt.savefig('fig_tracking.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_tracking.png')

## 8. Experiment 5: Signal Type Comparison

In [ ]:
np.random.seed(42)
T = 256
doas = [-20, 20]
K = 2
snr = 15

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
sig_types = [('bpsk', 'BPSK (κ=-2)'), ('qpsk', 'QPSK (κ=-1)'), ('gaussian', 'Gaussian (κ=0)')]

for ax, (stype, slabel) in zip(axes, sig_types):
    X = gen_signal(M, T, doas, snr=snr, signal_type=stype)
    C = compute_cumulant(X)

    P_mvdr = cop_mvdr_spectrum(C)
    P_sub = cop_subspace_spectrum(C, K)
    P_music = music_spectrum(X, K)

    ax.plot(scan_deg, 10*np.log10(P_music+1e-10), 'b-', lw=1.5, label='MUSIC')
    ax.plot(scan_deg, 10*np.log10(P_mvdr+1e-10), 'r-', lw=2, label='COP-MVDR')
    ax.plot(scan_deg, 10*np.log10(P_sub+1e-10), 'darkred', lw=2, ls='--', label='COP-Sub')
    for d in doas:
        ax.axvline(d, color='black', ls=':', alpha=0.5)
    ax.set_title(slabel, fontweight='bold')
    ax.set_xlim(-90, 90)
    ax.set_ylim(-30, 3)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('Spectrum (dB)')
axes[1].set_xlabel('DOA (degrees)')
fig.suptitle('COP Performance by Signal Type (κ = excess kurtosis)', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig_signal_types.png', dpi=150, bbox_inches='tight')
plt.show()
print('Note: Gaussian signals have κ=0, so cumulant=0 and COP fails → expected!')

## 9. Summary Table

In [ ]:
print('='*80)
print('  COP-RFS DOA Estimation: Method Comparison Summary')
print('='*80)
print(f'{"Method":>15s} | {"Array":>8s} | {"K needed":>8s} | {"Max K":>6s} | {"Noise":>12s} | Resolution')
print('-'*80)
rows = [
    ('CBF',       f'M={M}',   'No',   f'{M-1}',  'Sensitive',   'Low'),
    ('MVDR',      f'M={M}',   'No',   f'{M-1}',  'Sensitive',   'Medium'),
    ('MUSIC',     f'M={M}',   'Yes',  f'{M-1}',  'Sensitive',   'High'),
    ('COP-CBF',   f'M_v={M_V}', 'No',  f'{M_V-1}', 'Eliminated', 'Medium'),
    ('COP-MVDR',  f'M_v={M_V}', 'No',  f'{M_V-1}', 'Eliminated', 'High'),
    ('COP-Sub',   f'M_v={M_V}', 'Yes', f'{M_V-1}', 'Eliminated', 'Highest'),
]
for name, arr, kn, mk, noise, res in rows:
    kfree = ' *' if kn == 'No' else '  '
    print(f'{name:>15s} | {arr:>8s} | {kn:>8s}{kfree}| {mk:>6s} | {noise:>12s} | {res}')
print('-'*80)
print('* = K-free (practical advantage: no source count estimation needed)')
print(f'\nCOP-MVDR is the recommended K-free method for real-time tracking.')
print(f'COP-Sub gives highest resolution when K is known.')